# GPAT

Gridded Plume Analysis Tool (GPAT) modelling framework. This simulates flight trajectories, estimates fuel burn and emissions, models dispersion effects, and aggregates plume data to a common Eulerian grid for further photochemical and microphysical processing.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
from dataclasses import asdict
from pycontrails.models.gpat.gpat import GPAT, SimParams, FlParams, PlParams, MetParams, ChemParams, dict_to_dataclass
import os
import holoviews as hv
import hvplot.pandas
import hvplot.xarray

In [ ]:
# global simulation parameters
sim_params = {
    "t_fl": (pd.to_datetime("2022-01-20 13:00:00"), pd.Timedelta(minutes=1), pd.Timedelta(hours=1)),# (start time, time step, run time)
    "t_pl": (pd.to_datetime("2022-01-20 13:00:00"), pd.Timedelta(minutes=1), pd.Timedelta(hours=2)),# (start time, time step, max age)
    "t_sim": (pd.to_datetime("2022-01-20 12:00:00"), pd.Timedelta(seconds=20), pd.Timedelta(hours=4)),# (start time, time step, run time)
    "t_out": (pd.to_datetime("2022-01-20 12:00:00"), pd.Timedelta(minutes=1), pd.Timedelta(hours=4)),# (start time, time step, run time)
    "lat_bounds": (0.0, 1.0),  # lat bounds [deg]
    "lon_bounds": (0.0, 1.0),  # lon bounds [deg]
    "alt_bounds": (10000, 11000),  # alt bounds [m]
    "hres_sim_c": 0.05,  # coarse horizontal resolution [deg]
    "vres_sim_c": 500,  # coarse vertical resolution [m]
    "hres_sim_f": 0.001,  # fine horizontal resolution [deg]
    "vres_sim_f": 100,  # fine vertical resolution [m]

    "run_path": "/home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/",
    "data_path": "/home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/data/", # "/projects/Impact_of_aviation_on_climate
    "job_id": "GPAT_Feb_2026_test_1_ac",
}

In [ ]:
#flight trajectory parameters
fl_params = {
    "mode": "synthetic",
    "file": None,  # flight trajectory file

    "ac_type": "A320",  # aircraft type
    "fl0_speed": 150.0,  # m/s
    "fl0_heading": 45.0,  # deg
    "fl0_coords0": (0.1, 0.1, 10500),  # lat, lon, alt [deg, deg, m]
    "sep_dist": (10000, 5000, 0),  # dx, dy, dz [m]
    "n_ac": 2,  # number of aircraft
}

In [ ]:
# plume dispersion parameters
pl_params = {
    "depth": 50.0,  # initial plume depth, [m]
    "width": 50.0,  # initial plume width, [m]
    "verbose_outputs": False,  # print verbose outputs
    "shear": 0.01,  # shear [m/s]
    "n_slices": 3,  # number of slices in the plume
    "f_max": 0.99,  # maximum fraction of total emissions in any slice
    "output_pl_slices": True,  # output plume slices to netCDF
    }

In [ ]:
# meteorology parameters
met_params = {
    "eastward_wind": 5.0,  # m/s
    "northward_wind": 3.0,  # m/s
    "lagrangian_tendency_of_air_pressure": 0.0,  # m/s
}

In [ ]:
# chemistry parameters
chem_params = {
    "run_chem": True,
    "species_emi": ("NO", "CO", "SO2"),
    # "species_pl": ("NO", "CO", "SO2"),
    "species_pl": ("NO", "NO2", "O3", "NO3", "N2O5",
                      "HNO3", "HONO", "HO2NO2","PAN", 
                      "CH3O2NO2","H2O2", "CH3OOH",
                      "CO", "CH4", "HCHO", "SO2", "SA"),
    "species_out": ("O3", "NO2", "NO", "NO3", "N2O5", 
                    "HNO3", "HONO", "HO2", "OH", "H2O2",
                    "CO", "CH4", "CH3O2","HO2NO2", "PAN", "SO2" )
}

In [ ]:
sim_params = SimParams(**sim_params)
fl_params = FlParams(**fl_params)
pl_params = PlParams(**pl_params)
met_params = MetParams(**met_params)
chem_params = ChemParams(**chem_params)

gpat = GPAT(sim_params, fl_params, pl_params, met_params, chem_params)

In [ ]:
gpat.preprocess_gpat()

In [ ]:
gpat.eval()

In [ ]:
fl_ds = xr.open_dataset(f"{gpat.inputs_job}/fl_ds.nc")
fl_ds.flight_id.values

In [ ]:
from IPython.display import clear_output
clear_output(wait=True)

pl_ds = xr.open_dataset(f"{gpat.inputs_job}/pl_ds.nc")

pl_ds


In [ ]:
for t in pl_ds.time.values:
    active_mask = pl_ds["active_seg_flag"][:, pl_ds.get_index("time").get_loc(t)]
    active_seg_ids = pl_ds["seg_id"].values[active_mask.values]
    

# pl_ds.active_seg_flag.isel(seg_id=0).values

In [ ]:
# Matplotlib time slider (ipympl): trajectories + plume width (tail only)
%matplotlib widget
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import ipywidgets as widgets
from IPython.display import display


# gpat.analysis.plot_plumes()

In [ ]:
gpat.analysis.load_output_datasets()
# gpat.pl_out = xr.open_dataset(f"{gpat.outputs_job}/pl_out.nc")
gpat.pl_out
# gpat.pl_out.sel(slice_id=1, ht="tail", time='2022-01-20T13:40:00Z')["y_half"].values
# gpat.analysis.plot_plumes_3d_ani_plotly()

In [ ]:
from IPython.display import clear_output
clear_output(wait=True)

boxm_out = xr.open_dataset(f"{gpat.outputs_job}/boxm_out.nc")

boxm_out

In [ ]:
gpat.patch_table

In [ ]:
from IPython.display import clear_output
clear_output(wait=True)

pl_out = xr.open_dataset(f"{gpat.outputs_job}/pl_out.nc")
pl_out

In [ ]:
from IPython.display import clear_output
clear_output(wait=True)

np.set_printoptions(threshold=np.inf, linewidth=2000)
print(pl_out["pl_mass"].isel(species_pl=15, time=slice(60, 80)).values)

In [ ]:
pl_ds["species_emi_num"].values
pl_ds["emi_pl_mass"].isel(seg_id=0).to_pandas()
pl_ds["emi_pl_mass"].max("seg_id").to_pandas()

print(pl_ds["emi_pl_mass"].max("seg_id").to_pandas().to_string())

In [ ]:
pl_out["pl_mass"].max(("seg_id","time")).to_pandas()

In [ ]:
pd.DataFrame({
    "species_emi": pl_ds["species_emi"].values,
    "species_emi_num": pl_ds["species_emi_num"].values,
})

pd.DataFrame({
    "species_pl": pl_out["species_pl"].values,
    "species_pl_num": pl_out["species_pl_num"].values,
})

print(pd.DataFrame({
    "species_emi": pl_ds["species_emi"].values,
    "species_emi_num": pl_ds["species_emi_num"].values,
}).to_string(index=False))

print(pd.DataFrame({
    "species_pl": pl_out["species_pl"].values,
    "species_pl_num": pl_out["species_pl_num"].values,
}).to_string(index=False))